In [2]:
import torch
import torch.nn as nn
from torch_geometric.nn import GINConv, global_mean_pool, global_add_pool
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader



In [3]:

# ── 1. Load MUTAG ──────────────────────────────────────────────────────────────
dataset = TUDataset(root="data/TUDataset", name="MUTAG")

# Simple 80/20 train-test split (index-based, no shuffle for reproducibility)
n_train = int(0.8 * len(dataset))
train_dataset = dataset[:n_train]
test_dataset  = dataset[n_train:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train graphs: {len(train_dataset)}, Test graphs: {len(test_dataset)}")
print(f"Node features: {dataset.num_node_features}, Classes: {dataset.num_classes}")



Train graphs: 150, Test graphs: 38
Node features: 7, Classes: 2


In [4]:

# ── 2. GIN Encoder ─────────────────────────────────────────────────────────────
# We use GIN (Graph Isomorphism Network) layers - a strong and simple choice
# covered in the course. Each layer is:
#   h_v^{(k)} = MLP( h_v^{(k-1)} + sum_{u in N(v)} h_u^{(k-1)} )
#
# After L message-passing layers we do graph-level readout:
#   z_mean, z_log_var = MLP( mean_pool(h_v^{(L)}) )
# These are the VAE latent parameters used by Member 2.

class GINEncoder(nn.Module):
    """
    GIN-based encoder for a Graph VAE.

    Takes a batched graph (PyG Data/Batch) and returns:
        z_mean    : (batch_size, latent_dim)
        z_log_var : (batch_size, latent_dim)

    These are passed to Member 2's decoder to reconstruct adjacency matrices.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        latent_dim: int = 32,
        num_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        in_dim = input_dim
        for _ in range(num_layers):
            # Each GINConv wraps a 2-layer MLP
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            in_dim = hidden_dim

        self.dropout = nn.Dropout(dropout)

        # Readout MLP: graph embedding → VAE parameters
        self.fc_mean    = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, edge_index, batch):
        """
        Args:
            x          : node feature matrix  (total_nodes, input_dim)
            edge_index : edge connectivity     (2, total_edges)
            batch      : batch assignment      (total_nodes,)

        Returns:
            z_mean     : (batch_size, latent_dim)
            z_log_var  : (batch_size, latent_dim)
        """
        h = x
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)   # message passing
            h = bn(h)                 # normalise
            h = torch.relu(h)         # activate
            h = self.dropout(h)

        # Graph-level readout: average node embeddings per graph
        h_graph = global_mean_pool(h, batch)   # (batch_size, hidden_dim)

        z_mean    = self.fc_mean(h_graph)       # (batch_size, latent_dim)
        z_log_var = self.fc_log_var(h_graph)    # (batch_size, latent_dim)

        return z_mean, z_log_var



In [5]:

# ── 3. Quick sanity check ──────────────────────────────────────────────────────
if __name__ == "__main__":
    LATENT_DIM = 32

    encoder = GINEncoder(
        input_dim  = dataset.num_node_features,
        hidden_dim = 64,
        latent_dim = LATENT_DIM,
        num_layers = 3,
    )
    print(encoder)
    print(f"\nTrainable parameters: {sum(p.numel() for p in encoder.parameters()):,}")

    # Forward pass on one batch
    batch = next(iter(train_loader))
    z_mean, z_log_var = encoder(batch.x.float(), batch.edge_index, batch.batch)

    print(f"\nInput batch  : {batch.num_graphs} graphs")
    print(f"z_mean shape : {z_mean.shape}")       # → (batch_size, 32)
    print(f"z_log_var    : {z_log_var.shape}")    # → (batch_size, 32)
    print(f"z_mean range : [{z_mean.min():.3f}, {z_mean.max():.3f}]")


GINEncoder(
  (convs): ModuleList(
    (0): GINConv(nn=Sequential(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
    (1-2): 2 x GINConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (bns): ModuleList(
    (0-2): 3 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc_mean): Linear(in_features=64, out_features=32, bias=True)
  (fc_log_var): Linear(in_features=64, out_features=32, bias=True)
)

Trainable parameters: 25,859

Input batch  : 32 graphs
z_mean shape : torch.Size([32, 32])
z_log_var    : torch.Size([32, 32])
z_mean range : [-1.187, 1.871]


In [6]:
z_mean

tensor([[-0.3329, -0.2733,  0.0324,  ...,  0.0561, -0.2534, -0.0136],
        [-0.3211, -0.2662, -0.0602,  ..., -0.0049, -0.1719, -0.0459],
        [-0.5246, -0.2275, -0.3343,  ...,  0.0239, -0.0332,  0.0658],
        ...,
        [-0.2849, -0.3791, -0.0618,  ..., -0.1160, -0.3254, -0.0376],
        [-0.2853, -0.1525, -0.0442,  ..., -0.0791, -0.1660, -0.0735],
        [-0.3250, -0.0197,  0.0021,  ..., -0.0682, -0.3116, -0.1283]],
       grad_fn=<AddmmBackward0>)

In [7]:
z_log_var

tensor([[-0.2687,  0.3552,  0.3438,  ..., -0.3895, -0.2218, -0.1146],
        [-0.0280,  0.2479,  0.2547,  ..., -0.4793, -0.0798, -0.0588],
        [-0.4057,  0.1882,  0.4782,  ..., -0.0451, -0.2948, -0.0778],
        ...,
        [-0.5374,  0.4164,  0.3348,  ..., -0.1603, -0.3039, -0.1097],
        [-0.3344,  0.1579,  0.1249,  ..., -0.1981, -0.1796, -0.2269],
        [-0.1562,  0.1813,  0.1775,  ..., -0.4751, -0.1428, -0.1344]],
       grad_fn=<AddmmBackward0>)

In [8]:
# Save encoder to a .py file so others can import it
with open("gnn_encoder.py", "w") as f:
    f.write("""
import torch
import torch.nn as nn
from torch_geometric.nn import GINConv, global_mean_pool

class GINEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32, num_layers=3, dropout=0.1):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        in_dim = input_dim
        for _ in range(num_layers):
            mlp = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            in_dim = hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, edge_index, batch):
        h = x
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)
            h = bn(h)
            h = torch.relu(h)
            h = self.dropout(h)
        h_graph = global_mean_pool(h, batch)
        return self.fc_mean(h_graph), self.fc_log_var(h_graph)
""")